## combination of my scraper and romy's
This tests the first 50 people from the json file created by scraper 1. 


In [1]:
!pip install requests beautifulsoup4 tqdm

In [ ]:
import json
import time
import random
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime
import os
import csv

# ─────────────────────────────────────────────
## Change the name of the input file name to : hyrox_results_DUTCH_2026-11-20_16-17.json
# ─────────────────────────────────────────────
INPUT_FILE = "../data/raw/hyrox_results_DUTCH_2026-11-20_16-17.json" 

# ── Adjust this per computer ─────────────────
START = 0        
END   = 10
# ─────────────────────────────────────────────

OUTPUT_FILE = f"../data/raw/hyrox_detailed_{START}_{END}.json"
LOG_FILE    = f"../data/raw/hyrox_log_{START}_{END}.csv"

LOG_FIELDS = [
    "request_number",
    "timestamp",
    "input_row",
    "record_id",
    "url",
    "status_code",
    "success",
    "error_message"
]

WANTED_CLASSES = (
    [f"f-time_{i}" for i in range(11, 19)]
    + ["f-time_49", "f-time_50", "f-time_60"]
)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
]

def get_headers():
    return {"User-Agent": random.choice(USER_AGENTS)}

# ─────────────────────────────────────────────
# LOG HELPER
# ─────────────────────────────────────────────
def append_log(entry: dict):
    """Add one row to the CSV log file after each athlete."""
    file_exists = os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if not file_exists:
            writer.writeheader() 
        writer.writerow(entry)

# ─────────────────────────────────────────────
# SCRAPE FUNCTION
# ─────────────────────────────────────────────
def scrape_personal_page(url: str) -> dict:
    result = {
        "judging"        : {},
        "workout_summary": {},
        "scrape_error"   : None,
        "status_code"    : None
    }

    
# ── Request with retry ───────────────────
    for attempt in range(3):
        try:
            response = requests.get(url, headers=get_headers(), timeout=10)
            result["status_code"] = response.status_code

            if response.status_code == 403:
                time.sleep(random.uniform(10, 20))
                continue
            response.raise_for_status()
            break

        except requests.RequestException as e:
            result["scrape_error"] = str(e)
            if attempt == 2:
                return result
            time.sleep(random.uniform(5, 10))

    soup = BeautifulSoup(response.text, "html.parser")

    
# ── 1. Judging Decision ──────────────────
    judges_div = soup.find("div", id="detail-box-judges")
    if judges_div:
        for row in judges_div.find_all("tr"):
            label    = row.find("th", class_="desc")
            value_td = row.find("td")
            if label and value_td:
                key   = label.get_text(strip=True).replace("*", "").lower().replace(" ", "_")
                muted = value_td.find("span", class_="text-muted")
                value = None if muted else value_td.get_text(strip=True)
                result["judging"][key] = value
    else:
        result["scrape_error"] = "Judging section not found"

    
# ── 2. Workout Summary ───────────────────
    table = soup.select_one("#detail-box-other table.table")
    if table:
        for row in table.select("tbody tr"):
            row_classes = row.get("class", [])
            if any(cls in row_classes for cls in WANTED_CLASSES):
                cells = row.find_all(["th", "td"], recursive=False)
                if len(cells) == 3:
                    split_name = cells[0].get_text(strip=True)
                    time_val   = cells[1].get_text(strip=True)
                    place_raw  = cells[2].get_text(strip=True)
                    muted      = cells[2].find("span", class_="text-muted")
                    place_val  = None if muted else place_raw

                    result["workout_summary"][split_name] = {
                        "time" : time_val,
                        "place": int(place_val) if place_val and place_val.isdigit() else None
                    }
    else:
        result["scrape_error"] = "Workout table not found"

    return result

# ─────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────
def main():
    batch_start_time = datetime.now()
    print(f"Batch started   : {batch_start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        all_athletes = json.load(f)

    athletes = all_athletes[START:END]

    print(f"Total loaded    : {len(all_athletes)} athletes")
    print(f"Your batch      : [{START} → {END}] ({len(athletes)} athletes)\n")

    detailed_results = []
    request_number   = 0
    success_count    = 0
    failed_count     = 0

    for i, athlete in enumerate(tqdm(athletes, desc=f"Batch {START}-{END}", unit="athlete")):
        url       = athlete.get("personal_link")
        record_id = athlete.get("record_id")
        input_row = START + i

        request_number += 1
        timestamp = datetime.now().isoformat()

        if not url:
            append_log({
                "request_number": request_number,
                "timestamp"     : timestamp,
                "input_row"     : input_row,
                "record_id"     : record_id,
                "url"           : None,
                "status_code"   : None,
                "success"       : False,
                "error_message" : "No URL found"
            })
            athlete["detail_data"] = None
            detailed_results.append(athlete)
            failed_count += 1
            continue

        detail  = scrape_personal_page(url)
        success = detail["scrape_error"] is None

        
# ── Log row per athlete ──────────────
        append_log({
            "request_number": request_number,
            "timestamp"     : timestamp,
            "input_row"     : input_row,
            "record_id"     : record_id,
            "url"           : url,
            "status_code"   : detail["status_code"],
            "success"       : success,
            "error_message" : detail["scrape_error"] if not success else None
        })

        athlete["detail_data"] = {
            "judging"        : detail["judging"],
            "workout_summary": detail["workout_summary"]
        }
        athlete["scraped_at"] = timestamp
        detailed_results.append(athlete)

        if success:
            success_count += 1
        else:
            failed_count += 1

        time.sleep(random.uniform(2.0, 5.0))

    
# ── Final save ───────────────────────────
    batch_end_time = datetime.now()
    duration       = batch_end_time - batch_start_time
    total_minutes  = round(duration.total_seconds() / 60, 1)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(detailed_results, f, ensure_ascii=False, indent=2)

    print(f"\nDone! Saved to '{OUTPUT_FILE}'")
    print(f"Batch started   : {batch_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Batch finished  : {batch_end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total duration  : {total_minutes} minutes")
    print(f"Total requests  : {request_number}")
    print(f"Successful      : {success_count}")
    print(f"Failed          : {failed_count}")
    print(f"Log saved to    : '{LOG_FILE}'")

main()


Batch started   : 2026-11-27 12:25:11
Total loaded    : 14936 athletes
Your batch      : [0 → 10] (10 athletes)



Batch 0-10: 100%|██████████| 10/10 [00:38<00:00,  3.87s/athlete]


Done! Saved to '../data/raw/hyrox_detailed_0_10.json'
Batch started   : 2026-11-27 12:25:11
Batch finished  : 2026-11-27 12:25:49
Total duration  : 0.6 minutes
Total requests  : 10
Successful      : 10
Failed          : 0
Log saved to    : '../data/raw/hyrox_log_0_10.csv'
